In [ ]:
df['Local Time'] = pd.to_datetime(df['Local Time'])
df = df.set_index('Local Time').sort_index()

agg_map = {col: "mean" for col in df.columns}
if "IsHoliday" in agg_map:
    agg_map["IsHoliday"] = "max"

df = df.resample("3H").agg(agg_map).dropna().copy()

In [6]:
import pandas as pd
df = pd.read_csv(r"E:\Document\PROJECT\data\submission\data2225_done_model_ready.csv")
print(df.info())


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11529 entries, 0 to 11528
Data columns (total 39 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   Local Time                 11529 non-null  object 
 1   PM25_lag_1                 11529 non-null  float64
 2   PM25_lag_8                 11529 non-null  float64
 3   PM25_lag_24                11529 non-null  float64
 4   PM25_lag_56                11529 non-null  float64
 5   PM25_roll_mean_8           11529 non-null  float64
 6   PM25_roll_std_8            11529 non-null  float64
 7   PM25_roll_max_8            11529 non-null  float64
 8   PM25_roll_min_8            11529 non-null  float64
 9   PM25_roll_mean_24          11529 non-null  float64
 10  PM25_roll_std_24           11529 non-null  float64
 11  PM25_roll_max_24           11529 non-null  float64
 12  PM25_roll_min_24           11529 non-null  float64
 13  PM25_diff_1                11529 non-null  flo

In [9]:

df_2 = pd.read_csv(r"E:\Document\PROJECT\data\submission\data2225_done_cleaned.csv")
print(df_2.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 34759 entries, 0 to 34758
Data columns (total 16 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   Local Time                 34759 non-null  object 
 1   CO                         34759 non-null  float64
 2   NO2                        34759 non-null  float64
 3   O3                         34759 non-null  float64
 4   PM10                       34759 non-null  float64
 5   PM25                       34759 non-null  float64
 6   SO2                        34759 non-null  float64
 7   Clouds                     34759 non-null  int64  
 8   Precipitation              34759 non-null  float64
 9   Pressure                   34759 non-null  int64  
 10  Relative Humidity          34759 non-null  float64
 11  Temperature                34759 non-null  float64
 12  UV Index                   34759 non-null  float64
 13  Wind Speed                 34759 non-null  flo

In [ ]:
# ===== Time features =====
df["hour"] = df.index.hour
df["dayofweek"] = df.index.dayofweek   # Monday=0, Sunday=6
df["month"] = df.index.month

# ===== PM25 lag features =====
df["PM25_lag_1"] = df["PM25"].shift(1)
df["PM25_lag_8"] = df["PM25"].shift(8)
df["PM25_lag_24"] = df["PM25"].shift(24)

# bỏ các dòng đầu bị NaN do lag/rolling
df = df.dropna().copy()

print(df[[
    "PM25", "PM25_lag_1", "PM25_lag_8", "PM25_lag_24",
    "hour", "dayofweek", "month"
]].head())

In [ ]:
num_cols = df.select_dtypes(include=[np.number]).columns

skew_df = pd.DataFrame({
    "skew": df[num_cols].skew(numeric_only=True),
    "mean": df[num_cols].mean(numeric_only=True),
    "median": df[num_cols].median(numeric_only=True),
    "min": df[num_cols].min(numeric_only=True),
    "max": df[num_cols].max(numeric_only=True)
}).sort_values("skew", ascending=False)

skew_df["abs_skew"] = skew_df["skew"].abs()

strong_skew = skew_df[skew_df["abs_skew"] >= 1].sort_values("abs_skew", ascending=False)
moderate_skew = skew_df[(skew_df["abs_skew"] >= 0.5) & (skew_df["abs_skew"] < 1)]

print("=== Cột lệch mạnh ===")
print(strong_skew[["skew", "mean", "median", "min", "max"]])

print("\n=== Cột lệch vừa ===")
print(moderate_skew[["skew", "mean", "median", "min", "max"]])

In [ ]:
# đảm bảo index là datetime
df = df.copy()
df.index = pd.to_datetime(df.index)
df = df.sort_index()

# ===== Split theo tỷ lệ 70/15/15, giữ nguyên thứ tự thời gian =====
n_total = len(df)
train_size = int(n_total * 0.70)
val_size = int(n_total * 0.15)
test_size = n_total - train_size - val_size

train_df = df.iloc[:train_size].copy()
val_df = df.iloc[train_size:train_size + val_size].copy()
test_df = df.iloc[train_size + val_size:].copy()

train_end = train_df.index.max()
val_start = val_df.index.min()
val_end = val_df.index.max()
test_start = test_df.index.min()

print("Kích thước train/val/test:", train_df.shape, val_df.shape, test_df.shape)
print("Moc thoi gian:")
print("train:", train_df.index.min(), "->", train_end)
print("val  :", val_start, "->", val_end)
print("test :", test_start, "->", test_df.index.max())

In [ ]:
y_train_raw = train_df["PM25"].to_numpy()
y_val_raw = val_df["PM25"].to_numpy()
y_test_raw = test_df["PM25"].to_numpy()

train_log = np.log1p(train_df["PM25"])
val_log = np.log1p(val_df["PM25"])
test_log = np.log1p(test_df["PM25"])

y_train_t = train_log.to_numpy()
y_val_t = val_log.to_numpy()
y_test_t = test_log.to_numpy()

In [ ]:


df = df.copy()

df["hour"] = df.index.hour
df["dayofweek"] = df.index.dayofweek
df["month"] = df.index.month

df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24)
df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24)
df["dow_sin"] = np.sin(2 * np.pi * df["dayofweek"] / 7)
df["dow_cos"] = np.cos(2 * np.pi * df["dayofweek"] / 7)
df["month_sin"] = np.sin(2 * np.pi * df["month"] / 12)
df["month_cos"] = np.cos(2 * np.pi * df["month"] / 12)

for lag in [1, 8, 16, 24, 32, 40, 48, 56]:
    df[f"PM25_lag_{lag}"] = df["PM25"].shift(lag)

shifted = df["PM25"].shift(1)
df["PM25_roll_mean_8"] = shifted.rolling(window=8).mean()
df["PM25_roll_std_8"] = shifted.rolling(window=8).std()
df["PM25_roll_max_8"] = shifted.rolling(window=8).max()
df["PM25_roll_min_8"] = shifted.rolling(window=8).min()
df["PM25_roll_mean_24"] = shifted.rolling(window=24).mean()
df["PM25_roll_std_24"] = shifted.rolling(window=24).std()
df["PM25_roll_max_24"] = shifted.rolling(window=24).max()
df["PM25_roll_min_24"] = shifted.rolling(window=24).min()

df["PM25_diff_1"] = shifted.diff(1)
df["PM25_diff_8"] = shifted.diff(8)
same_hour_lags_3d = ["PM25_lag_8", "PM25_lag_16", "PM25_lag_24"]
same_hour_lags_7d = [
    "PM25_lag_8", "PM25_lag_16", "PM25_lag_24", "PM25_lag_32",
    "PM25_lag_40", "PM25_lag_48", "PM25_lag_56",
]
df["PM25_same_hour_mean_3d"] = df[same_hour_lags_3d].mean(axis=1)
df["PM25_same_hour_mean_7d"] = df[same_hour_lags_7d].mean(axis=1)
df["PM25_same_hour_std_7d"] = df[same_hour_lags_7d].std(axis=1)
df["PM25_same_hour_max_7d"] = df[same_hour_lags_7d].max(axis=1)

df = df.dropna().copy()

base_features = [
    "PM25_lag_1", "PM25_lag_8", "PM25_lag_24", "PM25_lag_56",
    "PM25_roll_mean_8", "PM25_roll_std_8", "PM25_roll_max_8", "PM25_roll_min_8",
    "PM25_roll_mean_24", "PM25_roll_std_24", "PM25_roll_max_24", "PM25_roll_min_24",
    "PM25_diff_1", "PM25_diff_8",
    "PM25_same_hour_mean_3d", "PM25_same_hour_mean_7d", "PM25_same_hour_std_7d", "PM25_same_hour_max_7d",
]

weather_features = [
    "Temperature", "Pressure", "Wind Speed",
    "Clouds", "Precipitation", "Relative Humidity",
    "Accumulated Hours of Rain",
]

pollution_features = ["PM10", "CO", "NO2", "O3", "SO2"]

calendar_features = [
    "hour_sin", "hour_cos",
    "dow_sin", "dow_cos",
    "month_sin", "month_cos",
    "IsHoliday",
]

manual_v1_features = (
    base_features
    + ["PM10", "CO", "NO2"]
    + ["Temperature", "Pressure", "Wind Speed", "Relative Humidity", "Precipitation", "Clouds"]
    + calendar_features
)

production_v1_features = base_features + calendar_features

feature_groups = {
    "base": base_features,
    "weather": weather_features,
    "pollution": pollution_features,
    "calendar": calendar_features,
}

candidate_feature_sets = {
    "base": base_features,
    "base_weather": base_features + weather_features,
    "base_weather_pollution": base_features + weather_features + pollution_features,
    "manual_v1": manual_v1_features,
    "production_v1": production_v1_features,
    "optimistic_v1": manual_v1_features,
    "all": base_features + weather_features + pollution_features + calendar_features,
}

GRA_POOL_NAME = "all"
GRA_TOP_K = 15
GRA_RHO = 0.5


def _minmax_01(values):
    values = np.asarray(values, dtype=np.float64).reshape(-1)
    vmin = np.nanmin(values)
    vmax = np.nanmax(values)
    if not np.isfinite(vmin) or not np.isfinite(vmax) or np.isclose(vmax, vmin):
        return np.zeros_like(values, dtype=np.float64)
    return (values - vmin) / (vmax - vmin)


def compute_gra_scores(X_df, y_series, rho=0.5):
    ref = _minmax_01(np.asarray(y_series, dtype=np.float64).reshape(-1))
    diffs = []
    normalized = {}

    for col in X_df.columns:
        seq = _minmax_01(X_df[col].to_numpy(dtype=np.float64))
        normalized[col] = seq
        diffs.append(np.abs(ref - seq))

    diff_matrix = np.vstack(diffs)
    delta_min = float(np.min(diff_matrix))
    delta_max = float(np.max(diff_matrix))
    if np.isclose(delta_max, 0.0):
        delta_max = 1.0

    rows = []
    for col in X_df.columns:
        diff = np.abs(ref - normalized[col])
        coeff = (delta_min + rho * delta_max) / (diff + rho * delta_max)
        rows.append({
            "feature": col,
            "gra_score": float(np.mean(coeff)),
        })

    return pd.DataFrame(rows).sort_values(["gra_score", "feature"], ascending=[False, True]).reset_index(drop=True)


for name, cols in candidate_feature_sets.items():
    missing_cols = [col for col in cols if col not in df.columns]
    if missing_cols:
        print(f"{name}: thiếu cột -> {missing_cols}")
    else:
        print(f"{name}: {len(cols)} đặc trưng")

feature_cols = candidate_feature_sets["manual_v1"]
print('Bộ đặc trưng đang dùng:', feature_cols)


In [ ]:
# CẤU HÌNH CHUNG CHO MÔ HÌNH

LOOKBACK = 112
CHUNK_HORIZON = 1
ROLLOUT_HORIZON = 40
N_CHUNKS = ROLLOUT_HORIZON // CHUNK_HORIZON
HORIZON = CHUNK_HORIZON
EVAL_SIZE = ROLLOUT_HORIZON
STEP_SIZE = ROLLOUT_HORIZON
STEP_HOURS = 3
MAX_FOLDS = 4
EPOCHS = 70
BATCH_SIZE = 64
SEED = 62
TARGET_TRANSFORM_MODE = "log1p"
SELECTED_FEATURE_SET = "all"
OPTIMISTIC_FEATURE_SET = "optimistic_v1"
INNER_VAL_SIZE = 56
PEAK_QUANTILE = 0.90
PEAK_WEIGHT = 2.5
HUBER_DELTA = 1.0
FINAL_EPOCH_FLOOR = 10
EARLY_STOPPING_PATIENCE = 24
LR_REDUCE_PATIENCE = 8
ROLLING_POLICY = "assimilated"
FEATURE_SET_COMPARE = [SELECTED_FEATURE_SET, OPTIMISTIC_FEATURE_SET]

random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

try:
    tf.config.experimental.enable_op_determinism()
except Exception as exc:
    print("Không bật được deterministic ops:", exc)

production_feature_cols = candidate_feature_sets[SELECTED_FEATURE_SET]
optimistic_feature_cols = candidate_feature_sets[OPTIMISTIC_FEATURE_SET]
feature_cols = production_feature_cols

print("Bộ đặc trưng production đang dùng:", SELECTED_FEATURE_SET)
print("Số lượng đặc trưng production:", len(production_feature_cols))
print("LOOKBACK / CHUNK_HORIZON / ROLLOUT_HORIZON:", LOOKBACK, CHUNK_HORIZON, ROLLOUT_HORIZON)
print("MAX_FOLDS, EPOCHS:", MAX_FOLDS, EPOCHS)
print("INNER_VAL_SIZE:", INNER_VAL_SIZE)
print("TARGET_TRANSFORM_MODE:", TARGET_TRANSFORM_MODE)
print("FINAL_EPOCH_FLOOR / EARLY_STOPPING_PATIENCE:", FINAL_EPOCH_FLOOR, EARLY_STOPPING_PATIENCE)


In [ ]:
# ===== Shared sequence helpers for all forecasting models =====

DECODER_FUTURE_COLS = [
    "hour_sin",
    "hour_cos",
    "dow_sin",
    "dow_cos",
    "month_sin",
    "month_cos",
    "IsHoliday",
]


def make_sequences(
    X,
    y,
    lookback,
    horizon,
    decoder_future=None,
):
    X = np.asarray(X, dtype=np.float32)
    y = np.asarray(y, dtype=np.float32).reshape(-1)

    if len(X) != len(y):
        raise ValueError(f"X and y must have the same length: len(X)={len(X)}, len(y)={len(y)}")

    if decoder_future is not None:
        decoder_future = np.asarray(decoder_future, dtype=np.float32)
        if len(decoder_future) != len(X):
            raise ValueError(
                "decoder_future must have the same length as X: "
                f"len(decoder_future)={len(decoder_future)}, len(X)={len(X)}"
            )
        decoder_width = 1 + decoder_future.shape[1]
    else:
        decoder_width = 1

    X_seq, decoder_seq, y_seq = [], [], []
    max_start = len(X) - lookback - horizon + 1
    if max_start <= 0:
        return (
            np.empty((0, lookback, X.shape[1]), dtype=np.float32),
            np.empty((0, horizon, decoder_width), dtype=np.float32),
            np.empty((0, horizon), dtype=np.float32),
        )

    for i in range(max_start):
        X_seq.append(X[i:i + lookback])
        y_future = y[i + lookback:i + lookback + horizon]

        last_y = y[i + lookback - 1]
        decoder_seed = np.full((horizon, 1), last_y, dtype=np.float32)

        if decoder_future is not None:
            future_cov = decoder_future[i + lookback:i + lookback + horizon]
            decoder_input = np.concatenate([decoder_seed, future_cov], axis=1)
        else:
            decoder_input = decoder_seed

        decoder_seq.append(decoder_input)
        y_seq.append(y_future)

    return (
        np.asarray(X_seq, dtype=np.float32),
        np.asarray(decoder_seq, dtype=np.float32),
        np.asarray(y_seq, dtype=np.float32),
    )


In [ ]:
# tính toán các metric MAE, RMSE, MAPE tổng thể và riêng cho các điểm đỉnh (peak) dựa trên ngưỡng quantile.
def compute_regression_metrics(y_true_flat, y_pred_flat, peak_quantile=0.90):
    y_true_flat = np.asarray(y_true_flat, dtype=np.float64).reshape(-1)
    y_pred_flat = np.asarray(y_pred_flat, dtype=np.float64).reshape(-1)

    metrics = {
        "mae": mean_absolute_error(y_true_flat, y_pred_flat),
        "mse": mean_squared_error(y_true_flat, y_pred_flat),
        "rmse": np.sqrt(mean_squared_error(y_true_flat, y_pred_flat)),
        "mape": np.mean(
            np.abs((y_true_flat - y_pred_flat) / np.clip(np.abs(y_true_flat), 1e-6, None))
        ) * 100,
    }

    peak_threshold = float(np.quantile(y_true_flat, peak_quantile))
    peak_mask = y_true_flat >= peak_threshold
    metrics["peak_threshold"] = peak_threshold
    metrics["peak_mae"] = mean_absolute_error(y_true_flat[peak_mask], y_pred_flat[peak_mask]) if np.any(peak_mask) else np.nan
    return metrics


def aggregate_multistep_forecasts(eval_index, y_true_2d, y_pred_2d):
    rows = []
    y_true_2d = np.asarray(y_true_2d, dtype=np.float64)
    y_pred_2d = np.asarray(y_pred_2d, dtype=np.float64)

    for seq_idx in range(y_pred_2d.shape[0]):
        for h in range(y_pred_2d.shape[1]):
            rows.append({
                "timestamp": eval_index[seq_idx + h],
                "horizon_step": h + 1,
                "y_true": y_true_2d[seq_idx, h],
                "y_pred": y_pred_2d[seq_idx, h],
            })

    long_df = pd.DataFrame(rows)
    agg_df = (
        long_df.groupby("timestamp", as_index=False)
        .agg(
            y_true=("y_true", "mean"),
            y_pred=("y_pred", "mean"),
            pred_std=("y_pred", "std"),
            n_votes=("y_pred", "size"),
        )
        .sort_values("timestamp")
    )
    agg_df["pred_std"] = agg_df["pred_std"].fillna(0.0)
    return long_df, agg_df


# Tính metric riêng cho từng horizon step.
def compute_horizon_step_metrics(long_df, peak_quantile=0.90):
    rows = []
    for horizon_step, step_df in long_df.groupby("horizon_step", sort=True):
        metrics = compute_regression_metrics(
            step_df["y_true"].to_numpy(),
            step_df["y_pred"].to_numpy(),
            peak_quantile=peak_quantile,
        )
        rows.append({
            "horizon_step": int(horizon_step),
            "n_points": int(len(step_df)),
            "mae": metrics["mae"],
            "mse": metrics["mse"],
            "rmse": metrics["rmse"],
            "mape": metrics["mape"],
            "peak_mae": metrics["peak_mae"],
            "peak_threshold": metrics["peak_threshold"],
        })
    return pd.DataFrame(rows)


# Gom toàn bộ quy trình đánh giá multi-step forecasting: từ long format, aggregate theo timestamp, tính metric theo horizon step, và metric tổng thể.
def summarize_multistep_predictions(eval_index, y_true_2d, y_pred_2d, peak_quantile=0.90):
    long_df, agg_df = aggregate_multistep_forecasts(eval_index, y_true_2d, y_pred_2d)
    horizon_metrics_df = compute_horizon_step_metrics(long_df, peak_quantile=peak_quantile)
    step1_df = long_df[long_df["horizon_step"] == 1].sort_values("timestamp").reset_index(drop=True)

    raw_metrics = compute_regression_metrics(
        np.asarray(y_true_2d, dtype=np.float64).reshape(-1),
        np.asarray(y_pred_2d, dtype=np.float64).reshape(-1),
        peak_quantile=peak_quantile,
    )
    step1_metrics = compute_regression_metrics(
        step1_df["y_true"].to_numpy(),
        step1_df["y_pred"].to_numpy(),
        peak_quantile=peak_quantile,
    )
    agg_metrics = compute_regression_metrics(
        agg_df["y_true"].to_numpy(),
        agg_df["y_pred"].to_numpy(),
        peak_quantile=peak_quantile,
    )

    return {
        "long_df": long_df,
        "agg_df": agg_df,
        "step1_df": step1_df,
        "horizon_metrics_df": horizon_metrics_df,
        "raw_metrics": raw_metrics,
        "step1_metrics": step1_metrics,
        "agg_metrics": agg_metrics,
    }



# ===== Biểu đồ zoom riêng cho từng mô hình trên tập test =====
def plot_model_zoom(timeline_df, model_label, color, days=14, step_hours=STEP_HOURS):
    plot_df = timeline_df[["timestamp", "y_true", "y_pred"]].copy()
    plot_df["timestamp"] = pd.to_datetime(plot_df["timestamp"])
    zoom_n = min(len(plot_df), int(days * 24 / step_hours))
    zoom_df = plot_df.tail(zoom_n).copy()

    plt.figure(figsize=(15, 5))
    plt.plot(
        zoom_df["timestamp"],
        zoom_df["y_true"],
        label="Thực tế (Test)",
        linewidth=1.9,
        color="#4C72B0",
    )
    plt.plot(
        zoom_df["timestamp"],
        zoom_df["y_pred"],
        label=model_label,
        linewidth=1.8,
        color=color,
    )
    plt.title(f"Biểu đồ zoom {zoom_n} mốc cuối của backtest {model_label}")
    plt.xlabel("Thời gian")
    plt.ylabel("PM2.5")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.gcf().autofmt_xdate()
    plt.tight_layout()
    plt.show()



In [ ]:

def transform_target(y_raw, scaler=None, fit=False, mode="log1p"):
    y_raw = np.asarray(y_raw, dtype=np.float64).reshape(-1)

    y_clip = np.clip(y_raw, 0.0, None)

    if mode == "log1p":
        y_t = np.log1p(y_clip).reshape(-1, 1)
    elif mode == "sqrt":
        y_t = np.sqrt(y_clip).reshape(-1, 1)
    elif mode == "raw":
        y_t = y_clip.reshape(-1, 1)
    else:
        raise ValueError(f"Chế độ biến đổi không hợp lệ: {mode}")

    if fit:
        scaler = StandardScaler()
        scaler.fit(y_t)
    elif scaler is None:
        raise ValueError("Khi fit=False, scaler không được để trống")

    y_scaled = scaler.transform(y_t).reshape(-1)
    return y_scaled, scaler


def inverse_target(y_scaled, scaler, mode="log1p"):
    y_scaled = np.asarray(y_scaled, dtype=np.float64)
    original_shape = y_scaled.shape
    y_unscaled = scaler.inverse_transform(y_scaled.reshape(-1, 1)).reshape(-1)

    if mode == "log1p":
        y_raw = np.expm1(y_unscaled)
    elif mode == "sqrt":
        y_raw = np.square(np.clip(y_unscaled, 0.0, None))
    elif mode == "raw":
        y_raw = y_unscaled
    else:
        raise ValueError(f"Chế độ biến đổi không hợp lệ: {mode}")

    y_raw = np.clip(y_raw, 0.0, None)
    return y_raw.reshape(original_shape)

In [ ]:
PRODUCTION_LAGS = [1, 8, 16, 24, 32, 40, 48, 56]

def build_history_feature_frame(raw_df):
    raw_df = raw_df.copy().sort_index()

    raw_df["hour"] = raw_df.index.hour
    raw_df["dayofweek"] = raw_df.index.dayofweek
    raw_df["month"] = raw_df.index.month
    raw_df["hour_sin"] = np.sin(2 * np.pi * raw_df["hour"] / 24)
    raw_df["hour_cos"] = np.cos(2 * np.pi * raw_df["hour"] / 24)
    raw_df["dow_sin"] = np.sin(2 * np.pi * raw_df["dayofweek"] / 7)
    raw_df["dow_cos"] = np.cos(2 * np.pi * raw_df["dayofweek"] / 7)
    raw_df["month_sin"] = np.sin(2 * np.pi * raw_df["month"] / 12)
    raw_df["month_cos"] = np.cos(2 * np.pi * raw_df["month"] / 12)

    for lag in PRODUCTION_LAGS:
        raw_df[f"PM25_lag_{lag}"] = raw_df["PM25"].shift(lag)

    shifted = raw_df["PM25"].shift(1)
    raw_df["PM25_roll_mean_8"] = shifted.rolling(window=8).mean()
    raw_df["PM25_roll_std_8"] = shifted.rolling(window=8).std()
    raw_df["PM25_roll_max_8"] = shifted.rolling(window=8).max()
    raw_df["PM25_roll_min_8"] = shifted.rolling(window=8).min()
    raw_df["PM25_roll_mean_24"] = shifted.rolling(window=24).mean()
    raw_df["PM25_roll_std_24"] = shifted.rolling(window=24).std()
    raw_df["PM25_roll_max_24"] = shifted.rolling(window=24).max()
    raw_df["PM25_roll_min_24"] = shifted.rolling(window=24).min()

    raw_df["PM25_diff_1"] = shifted.diff(1)
    raw_df["PM25_diff_8"] = shifted.diff(8)
    same_hour_lags_3d = ["PM25_lag_8", "PM25_lag_16", "PM25_lag_24"]
    same_hour_lags_7d = [
        "PM25_lag_8", "PM25_lag_16", "PM25_lag_24", "PM25_lag_32",
        "PM25_lag_40", "PM25_lag_48", "PM25_lag_56",
    ]
    raw_df["PM25_same_hour_mean_3d"] = raw_df[same_hour_lags_3d].mean(axis=1)
    raw_df["PM25_same_hour_mean_7d"] = raw_df[same_hour_lags_7d].mean(axis=1)
    raw_df["PM25_same_hour_std_7d"] = raw_df[same_hour_lags_7d].std(axis=1)
    raw_df["PM25_same_hour_max_7d"] = raw_df[same_hour_lags_7d].max(axis=1)
    return raw_df



def prepare_train_eval_sequences(
    train_X_df,
    train_y_df,
    eval_X_df,
    eval_y_df,
    lookback=72,
    horizon=72,
    target_mode="log1p",
    decoder_future_cols=None,
):
    if len(train_X_df) <= lookback:
        raise ValueError("train_X_df chỉ có {} mẫu, không đủ cho lookback={}".format(len(train_X_df), lookback))

    decoder_future_cols = [] if decoder_future_cols is None else list(decoder_future_cols)
    missing_train_decoder_cols = [col for col in decoder_future_cols if col not in train_X_df.columns]
    missing_eval_decoder_cols = [col for col in decoder_future_cols if col not in eval_X_df.columns]
    if missing_train_decoder_cols or missing_eval_decoder_cols:
        raise ValueError(
            "Missing decoder future cols: "
            f"train={missing_train_decoder_cols}, eval={missing_eval_decoder_cols}"
        )

    if list(train_X_df.columns) != list(eval_X_df.columns):
        raise ValueError("train_X_df and eval_X_df must have the same feature column order")

    x_scaler = StandardScaler()
    X_train_scaled = x_scaler.fit_transform(train_X_df.values)
    X_eval_scaled = x_scaler.transform(eval_X_df.values)

    decoder_future_idx = [train_X_df.columns.get_loc(col) for col in decoder_future_cols]
    train_decoder_future = X_train_scaled[:, decoder_future_idx] if decoder_future_idx else None
    eval_decoder_future = X_eval_scaled[:, decoder_future_idx] if decoder_future_idx else None

    y_train_scaled, y_scaler = transform_target(
        train_y_df.values.reshape(-1),
        scaler=None,
        fit=True,
        mode=target_mode,
    )
    y_eval_scaled, _ = transform_target(
        eval_y_df.values.reshape(-1),
        scaler=y_scaler,
        fit=False,
        mode=target_mode,
    )

    X_train_seq, decoder_train_seq, y_train_seq = make_sequences(
        X_train_scaled,
        y_train_scaled,
        lookback=lookback,
        horizon=horizon,
        decoder_future=train_decoder_future,
    )

    X_context = np.vstack([X_train_scaled[-lookback:], X_eval_scaled])
    y_context = np.concatenate([y_train_scaled[-lookback:], y_eval_scaled])

    decoder_future_context = None
    if decoder_future_idx:
        decoder_future_context = np.vstack([
            train_decoder_future[-lookback:],
            eval_decoder_future,
        ])

    X_eval_seq, decoder_eval_seq, y_eval_seq = make_sequences(
        X_context,
        y_context,
        lookback=lookback,
        horizon=horizon,
        decoder_future=decoder_future_context,
    )

    return (
        X_train_seq,
        decoder_train_seq,
        y_train_seq,
        X_eval_seq,
        decoder_eval_seq,
        y_eval_seq,
        x_scaler,
        y_scaler,
    )

def build_inference_inputs(history_raw_df, feature_cols, x_scaler, y_scaler, lookback=72, target_mode="log1p"):
    feature_frame = build_history_feature_frame(history_raw_df)
    feature_frame = feature_frame.dropna(subset=feature_cols + ["PM25"]).copy()
    if len(feature_frame) < lookback:
        raise ValueError("Không đủ lịch sử sau khi tạo feature cho input inference.")

    X_window = feature_frame[feature_cols].tail(lookback).to_numpy(dtype=np.float32)
    X_scaled = x_scaler.transform(X_window)
    last_target_scaled, _ = transform_target(
        np.array([feature_frame["PM25"].iloc[-1]], dtype=np.float32),
        scaler=y_scaler,
        fit=False,
        mode=target_mode,
    )
    return X_scaled[np.newaxis, ...], float(last_target_scaled[0])


def summarize_rollout_predictions(rollout_df, peak_quantile=0.90):
    rollout_df = rollout_df.copy().sort_values("timestamp").reset_index(drop=True)
    rollout_df["pred_std"] = 0.0

    chunk_rows = []
    for chunk_id, chunk_df in rollout_df.groupby("chunk_id", sort=True):
        metrics = compute_regression_metrics(
            chunk_df["y_true"].to_numpy(),
            chunk_df["y_pred"].to_numpy(),
            peak_quantile=peak_quantile,
        )
        chunk_rows.append({
            "chunk_id": int(chunk_id),
            "n_points": int(len(chunk_df)),
            "mae": metrics["mae"],
            "mse": metrics["mse"],
            "rmse": metrics["rmse"],
            "mape": metrics["mape"],
            "peak_mae": metrics["peak_mae"],
            "peak_threshold": metrics["peak_threshold"],
        })

    chunk_metrics_df = pd.DataFrame(chunk_rows)
    rollout_metrics = compute_regression_metrics(
        rollout_df["y_true"].to_numpy(),
        rollout_df["y_pred"].to_numpy(),
        peak_quantile=peak_quantile,
    )
    return {
        "timeline_df": rollout_df,
        "chunk_metrics_df": chunk_metrics_df,
        "rollout_metrics": rollout_metrics,
    }

def run_assimilated_rollout(
    model,
    history_raw_df,
    future_raw_df,
    feature_cols,
    x_scaler,
    y_scaler,
    lookback=72,
    chunk_horizon=24,
    rollout_horizon=72,
    target_mode="log1p",
    decoder_rollout="autoregressive",
):
    if decoder_rollout not in ("autoregressive", "constant_last"):
        raise ValueError('decoder_rollout phải là "autoregressive" hoặc "constant_last".')
    history_raw_df = history_raw_df.copy().sort_index()
    future_raw_df = future_raw_df.copy().sort_index().iloc[:rollout_horizon]
    if len(future_raw_df) == 0:
        raise ValueError("Không đủ dữ liệu tương lai để rollout theo yêu cầu.")

    rows = []
    for chunk_start in range(0, rollout_horizon, chunk_horizon):
        chunk_id = chunk_start // chunk_horizon + 1
        chunk_future = future_raw_df.iloc[chunk_start:chunk_start + chunk_horizon].copy()
        effective_chunk_horizon = len(chunk_future)
        if effective_chunk_horizon == 0:
            break

        X_input, last_target_scaled = build_inference_inputs(
            history_raw_df=history_raw_df,
            feature_cols=feature_cols,
            x_scaler=x_scaler,
            y_scaler=y_scaler,
            lookback=lookback,
            target_mode=target_mode,
        )

        if decoder_rollout == "constant_last":
            decoder_input = np.full((1, chunk_horizon, 1), last_target_scaled, dtype=np.float32)
            y_pred_scaled = model.predict([X_input, decoder_input], verbose=0)[0][:effective_chunk_horizon].astype(
                np.float32
            )
        else:
            decoder_input = np.zeros((1, chunk_horizon, 1), dtype=np.float32)
            decoder_input[0, 0, 0] = last_target_scaled
            y_pred_scaled = np.zeros((chunk_horizon,), dtype=np.float32)
            for step_idx in range(effective_chunk_horizon):
                decoder_forecast = model.predict([X_input, decoder_input], verbose=0)[0]
                y_pred_scaled[step_idx] = decoder_forecast[step_idx]
                if step_idx + 1 < chunk_horizon:
                    decoder_input[0, step_idx + 1, 0] = y_pred_scaled[step_idx]

        y_pred = inverse_target(
            y_pred_scaled[:effective_chunk_horizon],
            y_scaler,
            mode=target_mode,
        )

        for step_idx, (timestamp, y_true_value, y_pred_value) in enumerate(
            zip(chunk_future.index, chunk_future["PM25"].to_numpy(dtype=np.float64), y_pred),
            start=1,
        ):
            rows.append({
                "chunk_id": chunk_id,
                "chunk_step": step_idx,
                "global_step": chunk_start + step_idx,
                "timestamp": timestamp,
                "y_true": float(y_true_value),
                "y_pred": float(y_pred_value),
            })

        history_raw_df = pd.concat([history_raw_df, chunk_future], axis=0)

    rollout_df = pd.DataFrame(rows)
    return summarize_rollout_predictions(rollout_df, peak_quantile=PEAK_QUANTILE)


def run_recursive_rollout(
    model,
    history_raw_df,
    future_raw_df,
    feature_cols,
    x_scaler,
    y_scaler,
    lookback=72,
    chunk_horizon=24,
    rollout_horizon=72,
    target_mode="log1p",
    decoder_rollout="autoregressive",
):
    if decoder_rollout not in ("autoregressive", "constant_last"):
        raise ValueError('decoder_rollout phải là "autoregressive" hoặc "constant_last".')
    history_raw_df = history_raw_df.copy().sort_index()
    future_raw_df = future_raw_df.copy().sort_index().iloc[:rollout_horizon]
    if len(future_raw_df) == 0:
        raise ValueError("Không đủ dữ liệu tương lai để rollout theo yêu cầu.")

    rows = []
    for chunk_start in range(0, rollout_horizon, chunk_horizon):
        chunk_id = chunk_start // chunk_horizon + 1
        chunk_future = future_raw_df.iloc[chunk_start:chunk_start + chunk_horizon].copy()
        effective_chunk_horizon = len(chunk_future)
        if effective_chunk_horizon == 0:
            break

        X_input, last_target_scaled = build_inference_inputs(
            history_raw_df=history_raw_df,
            feature_cols=feature_cols,
            x_scaler=x_scaler,
            y_scaler=y_scaler,
            lookback=lookback,
            target_mode=target_mode,
        )

        if decoder_rollout == "constant_last":
            decoder_input = np.full((1, chunk_horizon, 1), last_target_scaled, dtype=np.float32)
            y_pred_scaled = model.predict([X_input, decoder_input], verbose=0)[0][:effective_chunk_horizon].astype(
                np.float32
            )
        else:
            decoder_input = np.zeros((1, chunk_horizon, 1), dtype=np.float32)
            decoder_input[0, 0, 0] = last_target_scaled
            y_pred_scaled = np.zeros((chunk_horizon,), dtype=np.float32)
            for step_idx in range(effective_chunk_horizon):
                decoder_forecast = model.predict([X_input, decoder_input], verbose=0)[0]
                y_pred_scaled[step_idx] = decoder_forecast[step_idx]
                if step_idx + 1 < chunk_horizon:
                    decoder_input[0, step_idx + 1, 0] = y_pred_scaled[step_idx]

        y_pred = inverse_target(
            y_pred_scaled[:effective_chunk_horizon],
            y_scaler,
            mode=target_mode,
        )

        for step_idx, (timestamp, y_true_value, y_pred_value) in enumerate(
            zip(chunk_future.index, chunk_future["PM25"].to_numpy(dtype=np.float64), y_pred),
            start=1,
        ):
            rows.append({
                "chunk_id": chunk_id,
                "chunk_step": step_idx,
                "global_step": chunk_start + step_idx,
                "timestamp": timestamp,
                "y_true": float(y_true_value),
                "y_pred": float(y_pred_value),
            })

        recursive_chunk = chunk_future.copy()
        recursive_chunk.loc[:, "PM25"] = np.asarray(y_pred, dtype=np.float64)
        history_raw_df = pd.concat([history_raw_df, recursive_chunk], axis=0)

    rollout_df = pd.DataFrame(rows)
    return summarize_rollout_predictions(rollout_df, peak_quantile=PEAK_QUANTILE)


def get_rollout_runner(policy):
    policy = str(policy).lower().strip()
    if policy == "assimilated":
        return run_assimilated_rollout
    if policy == "recursive":
        return run_recursive_rollout
    raise ValueError(f"Invalid ROLLING_POLICY: {policy}")



def run_backtest_over_full_test(
    model,
    history_raw_df,
    future_raw_df,
    feature_cols,
    x_scaler,
    y_scaler,
    lookback=72,
    chunk_horizon=24,
    rollout_horizon=72,
    target_mode="log1p",
    rolling_policy="assimilated",
    decoder_rollout="autoregressive",
):
    history_raw_df = history_raw_df.copy().sort_index()
    future_raw_df = future_raw_df.copy().sort_index()
    if len(future_raw_df) == 0:
        raise ValueError("Không đủ dữ liệu tương lai để rollout theo yêu cầu.")


    rollout_runner = get_rollout_runner(rolling_policy)

    timeline_frames = []
    chunk_metric_frames = []
    window_rows = []
    global_chunk_id = 0

    for window_id, start in enumerate(range(0, len(future_raw_df), rollout_horizon), start=1):
        window_future = future_raw_df.iloc[start:start + rollout_horizon].copy()
        effective_horizon = len(window_future)
        if effective_horizon == 0:
            break

        window_summary = rollout_runner(
            model=model,
            history_raw_df=history_raw_df,
            future_raw_df=window_future,
            feature_cols=feature_cols,
            x_scaler=x_scaler,
            y_scaler=y_scaler,
            lookback=lookback,
            chunk_horizon=chunk_horizon,
            rollout_horizon=effective_horizon,
            target_mode=target_mode,
            decoder_rollout=decoder_rollout,
        )

        window_timeline_df = window_summary["timeline_df"].copy()
        window_timeline_df["window_id"] = window_id
        window_timeline_df["window_start"] = window_future.index.min()
        window_timeline_df["window_end"] = window_future.index.max()
        window_timeline_df["global_test_step"] = np.arange(start + 1, start + len(window_timeline_df) + 1)
        timeline_frames.append(window_timeline_df)

        window_metrics = window_summary["rollout_metrics"]
        window_rows.append({
            "window_id": window_id,
            "window_start": window_future.index.min(),
            "window_end": window_future.index.max(),
            "n_points": len(window_timeline_df),
            "mae": window_metrics["mae"],
            "mse": window_metrics["mse"],
            "rmse": window_metrics["rmse"],
            "mape": window_metrics["mape"],
            "peak_mae": window_metrics["peak_mae"],
            "peak_threshold": window_metrics["peak_threshold"],
        })

        window_chunk_metrics_df = window_summary["chunk_metrics_df"].copy()
        window_chunk_metrics_df = window_chunk_metrics_df.rename(columns={"chunk_id": "chunk_id_within_window"})
        window_chunk_metrics_df["window_id"] = window_id
        window_chunk_metrics_df["window_start"] = window_future.index.min()
        window_chunk_metrics_df["window_end"] = window_future.index.max()
        window_chunk_metrics_df["global_chunk_id"] = np.arange(
            global_chunk_id + 1,
            global_chunk_id + len(window_chunk_metrics_df) + 1,
        )
        global_chunk_id += len(window_chunk_metrics_df)
        chunk_metric_frames.append(window_chunk_metrics_df)

        history_raw_df = pd.concat([history_raw_df, window_future], axis=0)

    if not timeline_frames:
        raise ValueError("Không tạo được cửa sổ backtest nào.")

    timeline_df = pd.concat(timeline_frames, ignore_index=True)
    timeline_df = timeline_df.sort_values("timestamp").reset_index(drop=True)
    chunk_metrics_df = pd.concat(chunk_metric_frames, ignore_index=True) if chunk_metric_frames else pd.DataFrame()
    window_metrics_df = pd.DataFrame(window_rows)
    overall_metrics = compute_regression_metrics(
        timeline_df["y_true"].to_numpy(),
        timeline_df["y_pred"].to_numpy(),
        peak_quantile=PEAK_QUANTILE,
    )
    return {
        "timeline_df": timeline_df,
        "chunk_metrics_df": chunk_metrics_df,
        "window_metrics_df": window_metrics_df,
        "rollout_metrics": overall_metrics,
    }


In [ ]:
def build_gru_model(
    lookback,
    n_features,
    horizon,
    gru_units=(128, 64),
    dense_units=128,
    dropout=0.2,
    recurrent_dropout=0.0,
    learning_rate=2e-4,
    loss_fn="mse",
    l2_reg=0.0,
    clipnorm=1.0,
    use_attention=False,
):
    if isinstance(gru_units, int):
        gru_units = (gru_units, max(gru_units // 2, 32))
    elif len(gru_units) == 1:
        gru_units = (gru_units[0], max(gru_units[0] // 2, 32))

    encoder_units = gru_units[0]
    decoder_units = tuple(gru_units[1:]) if len(gru_units) > 1 else (max(encoder_units // 2, 32),)
    decoder_last_units = decoder_units[-1]
    decoder_first_units = decoder_units[0]
    regularizer = tf.keras.regularizers.l2(l2_reg) if l2_reg and l2_reg > 0 else None

    encoder_inputs = tf.keras.layers.Input(shape=(lookback, n_features), name="encoder_inputs")
    decoder_inputs = tf.keras.layers.Input(shape=(horizon, 1), name="decoder_inputs")

    encoder_outputs, encoder_state = tf.keras.layers.GRU(
        encoder_units,
        return_sequences=True,
        return_state=True,
        dropout=dropout,
        recurrent_dropout=recurrent_dropout,
        kernel_regularizer=regularizer,
        name="encoder_gru",
    )(encoder_inputs)

    if encoder_units != decoder_first_units:
        decoder_initial_state = tf.keras.layers.Dense(
            decoder_first_units,
            activation="tanh",
            kernel_regularizer=regularizer,
            name="decoder_init_projection",
        )(encoder_state)
    else:
        decoder_initial_state = encoder_state

    x = decoder_inputs

    for i, units in enumerate(decoder_units, start=1):
        x = tf.keras.layers.GRU(
            units,
            return_sequences=True,
            dropout=dropout,
            recurrent_dropout=recurrent_dropout,
            kernel_regularizer=regularizer,
            name=f"decoder_gru_{i}",
        )(
            x,
            initial_state=[decoder_initial_state] if i == 1 else None,
        )

    decoder_outputs = x

    if use_attention:
        attention_values = encoder_outputs
        if encoder_units != decoder_last_units:
            attention_values = tf.keras.layers.Dense(
                decoder_last_units,
                kernel_regularizer=regularizer,
                name="encoder_attention_projection",
            )(attention_values)

        attention_context = tf.keras.layers.AdditiveAttention(name="temporal_attention")(
            [decoder_outputs, attention_values]
        )
        x = tf.keras.layers.Concatenate(name="decoder_attention_concat")(
            [decoder_outputs, attention_context]
        )
    else:
        x = decoder_outputs

    if dense_units:
        x = tf.keras.layers.TimeDistributed(
            tf.keras.layers.Dense(
                dense_units,
                activation="relu",
                kernel_initializer="he_normal",
                kernel_regularizer=regularizer,
            ),
            name="time_distributed_dense",
        )(x)
        x = tf.keras.layers.Dropout(dropout, name="decoder_dropout")(x)

    x = tf.keras.layers.TimeDistributed(
        tf.keras.layers.Dense(1),
        name="time_distributed_output",
    )(x)

        # ===== SỬA Ở ĐÂY =====
    forecast_delta = tf.keras.layers.Reshape(
        (horizon,), name="forecast_delta"
    )(x)
    decoder_baseline = tf.keras.layers.Reshape(
        (horizon,), name="decoder_baseline"
    )(decoder_inputs)
    outputs = tf.keras.layers.Add(
        name="forecast_output"
    )([forecast_delta, decoder_baseline])
    # =====================)

    model_name = "seq2seq_gru_attention" if use_attention else "seq2seq_gru"
    model = tf.keras.Model(inputs=[encoder_inputs, decoder_inputs], outputs=outputs, name=model_name)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate, clipnorm=clipnorm),
        loss=loss_fn,
        metrics=[tf.keras.metrics.MeanAbsoluteError(name="mae")],
    )
    return model


In [ ]:
def make_weighted_huber_loss(peak_threshold, peak_weight=2.0, delta=1.0, horizon=72):
    peak_threshold = tf.constant(float(peak_threshold), dtype=tf.float32)
    peak_weight = tf.constant(float(peak_weight), dtype=tf.float32)
    delta = tf.constant(float(delta), dtype=tf.float32)
    raw_step_weights = tf.reshape(tf.linspace(1.0, 1.8, horizon), (1, horizon))
    step_weights = raw_step_weights / tf.reduce_mean(raw_step_weights)

    def loss(y_true, y_pred):
        error = y_true - y_pred
        abs_error = tf.abs(error)
        huber = tf.where(
            abs_error <= delta,
            0.5 * tf.square(error),
            delta * (abs_error - 0.5 * delta),
        )
        peak_mask = tf.cast(y_true >= peak_threshold, tf.float32)
        peak_weights = 1.0 + peak_weight * peak_mask
        weighted = huber * peak_weights * step_weights
        return tf.reduce_mean(weighted)

    return loss

In [ ]:
df_raw_rollout = df.copy().sort_index()
df_wf = build_history_feature_frame(df_raw_rollout).dropna().copy()

train_wf = df_wf[:train_end].copy()
val_wf = df_wf[val_start:val_end].copy()
test_wf = df_wf[test_start:].copy()

train_raw = df_raw_rollout[:train_end].copy()
val_raw = df_raw_rollout[val_start:val_end].copy()
test_raw = df_raw_rollout[test_start:].copy()

for feature_set_name in [SELECTED_FEATURE_SET, OPTIMISTIC_FEATURE_SET]:
    local_cols = candidate_feature_sets[feature_set_name]
    missing_feature_cols = [c for c in local_cols if c not in df_wf.columns]
    if missing_feature_cols:
        raise ValueError(f"Thiếu các cột feature cho {feature_set_name}: {missing_feature_cols}")

MODEL_KWARGS = {
    "gru_units": (256, 128, 64),
    "dense_units": 128,
    "dropout": 0.15,
    "recurrent_dropout": 0.0,
    "learning_rate": 2e-4,
    "l2_reg": 1e-5,
    "clipnorm": 1.0,
    "use_attention": True,
}
MODEL_LABEL = "Seq2Seq GRU + Attention" if MODEL_KWARGS.get("use_attention", False) else "Seq2Seq GRU"

def make_callbacks():
    return [
        EarlyStopping(
            monitor="val_loss",
            patience=EARLY_STOPPING_PATIENCE,
            restore_best_weights=True,
            min_delta=1e-3,
        ),
        ReduceLROnPlateau(
            monitor="val_loss",
            factor=0.5,
            patience=LR_REDUCE_PATIENCE,
            min_lr=1e-6,
            verbose=1,
        ),
    ]


def resolve_final_epochs(best_epoch_local, floor=FINAL_EPOCH_FLOOR, ceiling=EPOCHS, multiplier=1.35):
    scaled_epoch = int(np.ceil(float(best_epoch_local) * float(multiplier)))
    return min(max(scaled_epoch, int(floor)), int(ceiling))


def split_train_inner_val(train_X_df, train_y_df, inner_val_size):
    min_train_rows = LOOKBACK + CHUNK_HORIZON
    if len(train_X_df) <= inner_val_size + min_train_rows:
        raise ValueError(
            f"Không đủ dữ liệu để tách inner val. Cần > {inner_val_size + min_train_rows} rows, nhận {len(train_X_df)}"
        )
    train_core_X = train_X_df.iloc[:-inner_val_size].copy()
    train_core_y = train_y_df.iloc[:-inner_val_size].copy()
    inner_val_X = train_X_df.iloc[-inner_val_size:].copy()
    inner_val_y = train_y_df.iloc[-inner_val_size:].copy()
    return train_core_X, train_core_y, inner_val_X, inner_val_y


def fit_selector_model(train_core_X, train_core_y, inner_val_X, inner_val_y, model_kwargs=None, epochs=EPOCHS):
    model_kwargs = MODEL_KWARGS if model_kwargs is None else model_kwargs
    (
        X_train_seq,
        decoder_train_seq,
        y_train_seq,
        X_inner_val_seq,
        decoder_inner_val_seq,
        y_inner_val_seq,
        x_scaler,
        y_scaler,
    ) = prepare_train_eval_sequences(
        train_core_X,
        train_core_y,
        inner_val_X,
        inner_val_y,
        lookback=LOOKBACK,
        horizon=CHUNK_HORIZON,
        target_mode=TARGET_TRANSFORM_MODE,
    )

    if len(X_train_seq) == 0 or len(X_inner_val_seq) == 0:
        raise ValueError("Không tạo được sequence cho train hoặc inner val. Kiểm tra lại kích thước dữ liệu và lookback/horizon.")

    peak_threshold = float(np.quantile(y_train_seq.reshape(-1), PEAK_QUANTILE))
    loss_fn = make_weighted_huber_loss(
        peak_threshold=peak_threshold,
        peak_weight=PEAK_WEIGHT,
        delta=HUBER_DELTA,
        horizon=CHUNK_HORIZON,
    )

    model = build_gru_model(
        lookback=LOOKBACK,
        n_features=X_train_seq.shape[2],
        horizon=CHUNK_HORIZON,
        loss_fn=loss_fn,
        **model_kwargs,
    )
    history = model.fit(
        [X_train_seq, decoder_train_seq],
        y_train_seq,
        validation_data=([X_inner_val_seq, decoder_inner_val_seq], y_inner_val_seq),
        epochs=epochs,
        batch_size=BATCH_SIZE,
        callbacks=make_callbacks(),
        verbose=0,
    )
    best_epoch = int(np.argmin(history.history["val_loss"])) + 1
    return model, history, best_epoch, x_scaler, y_scaler, peak_threshold


def fit_full_history_model(full_X_df, full_y_df, epochs, model_kwargs=None):
    model_kwargs = MODEL_KWARGS if model_kwargs is None else model_kwargs
    x_scaler = StandardScaler()
    X_full_scaled = x_scaler.fit_transform(full_X_df.values)
    y_full_scaled, y_scaler = transform_target(
        full_y_df.values.reshape(-1), scaler=None, fit=True, mode=TARGET_TRANSFORM_MODE
    )
    X_full_seq, decoder_full_seq, y_full_seq = make_sequences(
        X_full_scaled, y_full_scaled, lookback=LOOKBACK, horizon=CHUNK_HORIZON
    )

    if len(X_full_seq) == 0:
        raise ValueError("Không tạo được sequence full-history")

    peak_threshold = float(np.quantile(y_full_seq.reshape(-1), PEAK_QUANTILE))
    loss_fn = make_weighted_huber_loss(
        peak_threshold=peak_threshold,
        peak_weight=PEAK_WEIGHT,
        delta=HUBER_DELTA,
        horizon=CHUNK_HORIZON,
    )

    model = build_gru_model(
        lookback=LOOKBACK,
        n_features=X_full_seq.shape[2],
        horizon=CHUNK_HORIZON,
        loss_fn=loss_fn,
        **model_kwargs,
    )
    model.fit(
        [X_full_seq, decoder_full_seq],
        y_full_seq,
        epochs=epochs,
        batch_size=BATCH_SIZE,
        verbose=0,
    )
    return model, x_scaler, y_scaler, peak_threshold


def fit_test_variant(feature_cols_local, history_feature_df, history_raw_df, future_raw_df, model_kwargs, selector_epochs=EPOCHS):
    history_X_df = history_feature_df[feature_cols_local].copy()
    history_y_df = history_feature_df[["PM25"]].copy()

    train_core_X, train_core_y, inner_val_X, inner_val_y = split_train_inner_val(
        history_X_df, history_y_df, INNER_VAL_SIZE
    )
    selector_model, selector_history, best_epoch_local, _, _, peak_threshold_train = fit_selector_model(
        train_core_X,
        train_core_y,
        inner_val_X,
        inner_val_y,
        model_kwargs=model_kwargs,
        epochs=selector_epochs,
    )
    final_epochs_local = resolve_final_epochs(best_epoch_local)
    print(f"[GRU] selector best_epoch={best_epoch_local}, final_epochs={final_epochs_local}")
    final_model, x_scaler_local, y_scaler_local, peak_threshold_full = fit_full_history_model(
        history_X_df,
        history_y_df,
        final_epochs_local,
        model_kwargs=model_kwargs,
    )
    rollout_summary = run_backtest_over_full_test(
        model=final_model,
        history_raw_df=history_raw_df,
        future_raw_df=future_raw_df,
        feature_cols=feature_cols_local,
        x_scaler=x_scaler_local,
        y_scaler=y_scaler_local,
        lookback=LOOKBACK,
        chunk_horizon=CHUNK_HORIZON,
        rollout_horizon=ROLLOUT_HORIZON,
        target_mode=TARGET_TRANSFORM_MODE,
    )
    return {
        "selector_model": selector_model,
        "selector_history": selector_history,
        "best_epoch": best_epoch_local,
        "final_epochs": final_epochs_local,
        "train_peak_threshold_t": peak_threshold_train,
        "train_full_peak_threshold_t": peak_threshold_full,
        "rollout_summary": rollout_summary,
        "final_model": final_model,
        "x_scaler": x_scaler_local,
        "y_scaler": y_scaler_local,
        "feature_cols": feature_cols_local,
    }


val_feature_pool = val_wf[feature_cols].copy()
base_train_X = train_wf[feature_cols].copy()
base_train_y = train_wf[["PM25"]].copy()

fold_rows = []
compare_samples = []

for fold, start in enumerate(range(0, len(val_feature_pool) - ROLLOUT_HORIZON + 1, STEP_SIZE), start=1):
    if fold > MAX_FOLDS:
        break

    end = start + ROLLOUT_HORIZON
    fold_eval_raw = val_raw.iloc[start:end].copy()
    if len(fold_eval_raw) < ROLLOUT_HORIZON:
        break

    fold_history_X = pd.concat([base_train_X, val_feature_pool.iloc[:start]], axis=0)
    fold_history_y = pd.concat([base_train_y, val_wf.iloc[:start][["PM25"]]], axis=0)
    train_core_X, train_core_y, inner_val_X, inner_val_y = split_train_inner_val(
        fold_history_X, fold_history_y, INNER_VAL_SIZE
    )

    model, history, best_epoch, x_scaler, y_scaler, peak_threshold_train = fit_selector_model(
        train_core_X, train_core_y, inner_val_X, inner_val_y
    )

    fold_history_raw = pd.concat([train_raw, val_raw.iloc[:start]], axis=0)
    rollout_summary = run_assimilated_rollout(
    model=model,
    history_raw_df=fold_history_raw,
    future_raw_df=fold_eval_raw,
    feature_cols=feature_cols,
    x_scaler=x_scaler,
    y_scaler=y_scaler,
    lookback=LOOKBACK,
    chunk_horizon=CHUNK_HORIZON,
    rollout_horizon=ROLLOUT_HORIZON,
    target_mode=TARGET_TRANSFORM_MODE,
)

    chunk_metrics_df = rollout_summary["chunk_metrics_df"].set_index("chunk_id")
    rollout_metrics = rollout_summary["rollout_metrics"]
    fold_record = {
        "fold": fold,
        "train_rows": len(fold_history_X),
        "inner_val_rows": len(inner_val_X),
        "rollout_rows": len(fold_eval_raw),
        "rollout_mae": rollout_metrics["mae"],
        "rollout_mse": rollout_metrics["mse"],
        "rollout_rmse": rollout_metrics["rmse"],
        "rollout_mape": rollout_metrics["mape"],
        "rollout_peak_mae": rollout_metrics["peak_mae"],
        "rollout_peak_threshold": rollout_metrics["peak_threshold"],
        "train_peak_threshold_t": peak_threshold_train,
        "best_epoch": best_epoch,
        "best_inner_val_loss": float(np.min(history.history["val_loss"])),
    }

    for chunk_id in range(1, N_CHUNKS + 1):
        chunk_metrics = chunk_metrics_df.loc[chunk_id]
        fold_record[f"chunk_{chunk_id}_mae"] = chunk_metrics["mae"]
        fold_record[f"chunk_{chunk_id}_rmse"] = chunk_metrics["rmse"]
        fold_record[f"chunk_{chunk_id}_mape"] = chunk_metrics["mape"]
        fold_record[f"chunk_{chunk_id}_peak_mae"] = chunk_metrics["peak_mae"]

    compare_samples.append({
        "fold": fold,
        "timeline_df": rollout_summary["timeline_df"].copy(),
        "chunk_metrics_df": rollout_summary["chunk_metrics_df"].copy(),
    })
    fold_rows.append(fold_record)

walkforward_df = pd.DataFrame(fold_rows)
print(f"=== Đánh giá walk-forward: rollout mô phỏng triển khai {ROLLOUT_HORIZON} bước = {CHUNK_HORIZON} x {N_CHUNKS} ===")
display(walkforward_df)

if not walkforward_df.empty:
    val_summary = {
        "mean_rollout_mae": walkforward_df["rollout_mae"].mean(),
        "mean_rollout_mse": walkforward_df["rollout_mse"].mean(),
        "mean_rollout_rmse": walkforward_df["rollout_rmse"].mean(),
    }

    chunk_mae_cols = sorted(
        [
            col for col in walkforward_df.columns
            if len(col.split("_")) == 3 and col.startswith("chunk_") and col.endswith("_mae")
        ],
        key=lambda col: int(col.split("_")[1]),
    )

    for col in chunk_mae_cols:
        chunk_id = int(col.split("_")[1])
        val_summary[f"mean_chunk_{chunk_id}_mae"] = walkforward_df[col].mean()

    val_summary_df = pd.DataFrame([val_summary])
    print("\n=== Bảng tóm tắt validation ===")
    display(val_summary_df)


# ===== Train trên train+val và backtest trên toàn bộ test =====
DEFAULT_ARTIFACT_ROOT = "/content/drive/MyDrive/pm25_model_registry" if __import__("pathlib").Path("/content/drive/MyDrive").exists() else "/content/model_registry"

def save_variant_bundle(variant, model_name, metrics, bundle_key, export_root=DEFAULT_ARTIFACT_ROOT):
    import json
    import pickle
    import shutil
    from pathlib import Path

    required_keys = ["final_model", "x_scaler", "y_scaler", "feature_cols", "train_full_peak_threshold_t"]
    missing_keys = [key for key in required_keys if key not in variant]
    if missing_keys:
        raise ValueError(f"Variant của {model_name} thiếu artifact: {missing_keys}")

    export_root = Path(export_root)
    bundle_dir = export_root / bundle_key
    if bundle_dir.exists():
        shutil.rmtree(bundle_dir)
    bundle_dir.mkdir(parents=True, exist_ok=True)

    variant["final_model"].save(bundle_dir / "model.keras")
    with open(bundle_dir / "x_scaler.pkl", "wb") as f:
        pickle.dump(variant["x_scaler"], f)
    with open(bundle_dir / "y_scaler.pkl", "wb") as f:
        pickle.dump(variant["y_scaler"], f)
    with open(bundle_dir / "feature_cols.pkl", "wb") as f:
        pickle.dump(list(variant["feature_cols"]), f)

    config = {
        "model_name": model_name,
        "bundle_key": bundle_key,
        "lookback": int(LOOKBACK),
        "chunk_horizon": int(CHUNK_HORIZON),
        "rollout_horizon": int(ROLLOUT_HORIZON),
        "target_transform_mode": TARGET_TRANSFORM_MODE,
        "feature_cols": list(variant["feature_cols"]),
        "best_epoch": int(variant["best_epoch"]),
        "final_epochs": int(variant.get("final_epochs", variant["best_epoch"])),
        "peak_threshold": float(variant["train_full_peak_threshold_t"]),
    }
    if variant.get("decoder_future_cols"):
        config["decoder_future_cols"] = list(variant["decoder_future_cols"])
    with open(bundle_dir / "config.json", "w", encoding="utf-8") as f:
        json.dump(config, f, ensure_ascii=False, indent=2)

    metrics_payload = {
        key: (float(value) if hasattr(value, "__float__") else value)
        for key, value in metrics.items()
    }
    with open(bundle_dir / "metrics.json", "w", encoding="utf-8") as f:
        json.dump(metrics_payload, f, ensure_ascii=False, indent=2)

    if "rollout_summary" in variant and "timeline_df" in variant["rollout_summary"]:
        variant["rollout_summary"]["timeline_df"].to_csv(bundle_dir / "test_timeline.csv", index=False)

    zip_path = shutil.make_archive(str(bundle_dir), "zip", bundle_dir)
    print(f"Saved bundle for {model_name}: {bundle_dir}")
    return str(bundle_dir), zip_path

train_val_feature_df = pd.concat([train_wf, val_wf], axis=0)
train_val_raw = pd.concat([train_raw, val_raw], axis=0)
test_backtest_raw = test_raw.copy()
test_rollout_raw = test_backtest_raw.copy()

if len(test_backtest_raw) < CHUNK_HORIZON:
    raise ValueError("Không đủ dữ liệu test để backtest.")

test_variant = fit_test_variant(
    feature_cols_local=feature_cols,
    history_feature_df=train_val_feature_df,
    history_raw_df=train_val_raw,
    future_raw_df=test_backtest_raw,
    model_kwargs=MODEL_KWARGS,
    selector_epochs=EPOCHS,
)

best_epoch_test = test_variant["best_epoch"]
peak_threshold_test_train = test_variant["train_peak_threshold_t"]
peak_threshold_test_full = test_variant["train_full_peak_threshold_t"]
test_eval_summary = test_variant["rollout_summary"]
test_timeline_df = test_eval_summary["timeline_df"].copy()
test_chunk_metrics_df = test_eval_summary["chunk_metrics_df"].copy()
test_window_metrics_df = test_eval_summary["window_metrics_df"].copy()
test_rollout_metrics = test_eval_summary["rollout_metrics"]
test_bundle_dir, test_bundle_zip = save_variant_bundle(
    variant=test_variant,
    model_name=MODEL_LABEL,
    metrics=test_rollout_metrics,
    bundle_key="seq2seq_gru_attention" if MODEL_KWARGS.get("use_attention", False) else "seq2seq_gru",
)

test_metrics_df = pd.DataFrame([
    {
        "best_epoch": best_epoch_test,
        "test_rows": len(test_timeline_df),
        "n_backtest_windows": test_window_metrics_df["window_id"].nunique(),
        "window_horizon": ROLLOUT_HORIZON,
        "chunk_horizon": CHUNK_HORIZON,
        "test_mae": test_rollout_metrics["mae"],
        "test_mse": test_rollout_metrics["mse"],
        "test_rmse": test_rollout_metrics["rmse"],
        "test_mape": test_rollout_metrics["mape"],
        "test_peak_mae": test_rollout_metrics["peak_mae"],
        "train_peak_threshold_t": peak_threshold_test_train,
        "train_val_peak_threshold_t": peak_threshold_test_full,
    }
])

test_report_df = pd.DataFrame([
    {
        "model": MODEL_LABEL,
        "test_rows": len(test_timeline_df),
        "n_backtest_windows": test_window_metrics_df["window_id"].nunique(),
        "window_horizon": ROLLOUT_HORIZON,
        "chunk_horizon": CHUNK_HORIZON,
        "mae": test_rollout_metrics["mae"],
        "mse": test_rollout_metrics["mse"],
        "rmse": test_rollout_metrics["rmse"],
        "MAPE": test_rollout_metrics["mape"],
        "peak_mae": test_rollout_metrics["peak_mae"],
        "best_epoch": best_epoch_test,
    }
])

print("\n=== Chỉ số backtest trên toàn bộ tập test ===")
display(test_metrics_df)

print("\n=== Chỉ số theo từng cửa sổ backtest ===")
display(test_window_metrics_df)

print("\n=== Chỉ số theo từng chunk trong backtest ===")
display(test_chunk_metrics_df)

print("\n=== Bảng tổng hợp kết quả ===")
display(test_report_df)

plot_df = test_timeline_df.copy()

zoom_n = min(len(plot_df), int(14 * 24 / STEP_HOURS))
zoom_hours = zoom_n * STEP_HOURS
zoom_days = zoom_hours / 24
zoom_plot_df = plot_df.tail(zoom_n).copy()

plt.figure(figsize=(15, 5))
plt.plot(
    zoom_plot_df["timestamp"],
    zoom_plot_df["y_true"],
    label="Thực tế (Test)",
    linewidth=1.9,
    color="#4C72B0",
)
plt.plot(
    zoom_plot_df["timestamp"],
    zoom_plot_df["y_pred"],
    label=MODEL_LABEL,
    linewidth=1.8,
    linestyle="--",
    color="#DD8452",
)
plt.title(f"Dự báo {MODEL_LABEL} trên {zoom_days:.0f} ngày cuối của tập test ({zoom_hours} giờ)")
plt.xlabel("Thời gian")
plt.ylabel("PM2.5")
plt.legend()
plt.grid(alpha=0.3)
plt.gcf().autofmt_xdate()
plt.tight_layout()
plt.show()
